# Lesson 9.4: Turn a plan into a bounded investigation

Read the matching lesson at `modules/09-agent-loops/04-plans-budgets-and-evidence.md`. Run this notebook from top to bottom in the project `.venv` kernel. All cases are fictional. The default path sends no model-provider requests. A kernel retains values between cells; restart and run all after an experiment to check independence.

## The practical question

An agent loop chooses another step after observing an earlier result. That flexibility can help an investigation, but also creates repeated calls, unexpected branches and uncertain completion. The application must bound the execution independently of the planner.

In [ ]:
from ai_orgdesign.labs import m09 as lab
print("Module 9 code loaded; no provider called.")

## Follow the mechanism

run receives a planner function instead of constructing a hidden model client. Each iteration supplies accumulated observations and expects either a final answer or a structured tool request. The application validates the decision, rejects repeated call identifiers, invokes the same scoped dispatcher used in Module 2 and records the observation. A positive integer budget bounds iterations. scripted_planner supplies deterministic decisions for reproducible tests; it is not called an LLM. The optional live adapter in ai_orgdesign.live uses the provider protocol while retaining the dispatcher and a separate tool-call limit.

In [ ]:
import inspect
print(inspect.getsource(lab.run))

In [ ]:
import inspect
print(inspect.getsource(lab.scripted_planner))

## Predict and run

The supplied planner first reads H-17 and then returns an investigation statement. With a budget of one, the tool can run but the loop stops at budget before a final answer. Repeated call identifiers stop the loop instead of executing indefinitely. Reaching a stop condition does not mean the business case is resolved.

In [ ]:
from ai_orgdesign.labs.m09 import run, scripted_planner
assert run(scripted_planner)["stop"] == "final"
assert run(scripted_planner, budget=1)["stop"] == "budget"
print(run(scripted_planner))

## Exercise

Supply a planner that always asks for the same call. Then change the call identifier on every step. Which independent bound remains necessary?

Pause and write the predicted result and its interpretation before opening the worked answer below.

## Worked answer

The repeated-identifier check catches the first planner. Unique identifiers evade that check, so the iteration budget remains essential. A live service also needs timeouts, token and cost limits, cancellation and recovery. Tool checks still apply on every execution; a model instruction to behave carefully is not a substitute.

In [ ]:
from ai_orgdesign.labs.m09 import run
request = {"kind":"tool", "id":"one", "name":"lookup_case", "arguments":{"case_id":"H-17"}}
assert run(lambda observations: request)["stop"] == "repeated_call"
print("Repeated call stopped")

## Alternatives and organisational meaning

A fixed workflow is often easier to verify when the route is known. An adaptive loop is justified when observations genuinely determine the next action. Framework runners can implement the loop, but the application still owns permissions, budgets and accepted completion.

Explain separately what executed, what its output establishes and what remains a question for the surrounding service.